In [1]:
# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import pandas
import torch
import pandas as pd
import numpy as np
from torch import nn, optim
import advanced_training_options
device = advanced_training_options.assess_device()
print(device)

mps


# Abalone
Predict the rings (age) of an abalone (orecchie di mare) from it's measurements

In [2]:
data_dict = advanced_training_options.read_split_abalone(device=device)


In [3]:
df = data_dict['df']
feature_columns = data_dict['feature_columns']
scaler = data_dict['scaler']
x_train = data_dict['x_train']
y_train = data_dict['y_train']
x_val = data_dict['x_val']
y_val = data_dict['y_val']
x_test = data_dict['x_test']
y_test = data_dict['y_test']
train_loader = data_dict['train_loader']
val_loader = data_dict['val_loader']
test_loader = data_dict['test_loader']
batch_size = data_dict['batch_size']
num_feats = data_dict['num_feats']

In [4]:
base_model = advanced_training_options.AbaloneNet(num_feats)
base_model.to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(params=base_model.parameters(), lr=0.01)

m_res, hist, b_ep = advanced_training_options.basic_training_loop(base_model,train_loader, loss_function, optimizer)

In [5]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res,x_test,y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)

Evaluation
MSE:  6.1895
RMSE: 2.4879 rings
MAE:  1.6968 rings
R^2:  0.3970


,actual_rings,predicted_rings
0,8.0,9.0
1,11.0,15.0
2,22.0,14.0
3,7.0,7.0
4,9.0,10.0
5,10.0,12.0
6,13.0,12.0
7,6.0,6.0
8,9.0,9.0
9,8.0,8.0


# Early Stopping
Early stopping is a technique that helps prevent overfitting and optimize model performance by monitoring validation loss during training. 

Overfitting will occur if you train the neural network for too many epochs, and the neural network will not perform well on new data, despite attaining a good accuracy on the training set. Overfitting occurs when a neural network is trained to the point that it begins to memorize rather than generalize

![Training vs. Validation Error for Overfitting](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_3_training_val.png "Training vs. Validation Error for Overfitting")



In [6]:
base_model = advanced_training_options.AbaloneNet(num_feats)
base_model.to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(params=base_model.parameters(), lr=0.01)

m_res, hist, b_ep = advanced_training_options.basic_training_loop(base_model,train_loader, loss_function, optimizer, patience=30, min_delta=0.001, val_loader=val_loader)

Epoch 25/500, Train Loss: 4.5025, Validation Loss: 4.6007, Validation MAE: 1.5156
Epoch 50/500, Train Loss: 4.2806, Validation Loss: 4.6585, Validation MAE: 1.5196
Stopping at epoch 57. Early stopping triggered after 30 epochs.


In [7]:
b_ep

27

In [8]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res,x_test,y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)

Evaluation
MSE:  4.0752
RMSE: 2.0187 rings
MAE:  1.4344 rings
R^2:  0.6030


,actual_rings,predicted_rings
0,8.0,9.0
1,11.0,10.0
2,22.0,21.0
3,7.0,8.0
4,9.0,10.0
5,10.0,13.0
6,13.0,12.0
7,6.0,9.0
8,9.0,10.0
9,8.0,9.0


# Training Schedules

Learning rate schedules are mechanisms used during the training of neural networks to adjust the learning rate over time. They're designed to decrease the learning rate as the training progresses, allowing the network to make large adjustments in the initial stages of training, when the weights are likely far from their optimal values, and then make smaller adjustments as the training progresses, to fine-tune the weights.

In PyTorch, one of the learning rate scheduling tools is the StepLR class, found in the **torch.optim.lr_scheduler** module. **StepLR** is a type of learning rate schedule that decreases the learning rate by a certain factor every few epochs.

StepLR takes three parameters:

* **optimizer:** The optimizer you're using to train your model (e.g., SGD, Adam).
* **step_size:** This is the number of epochs after which you want to reduce the learning rate. For instance, if step_size=10, then the learning rate will be reduced every 10 epochs.
* **gamma:** This is the factor by which the learning rate will be reduced at each step. For instance, if gamma=0.1, the learning rate will be multiplied by 0.1 at each step, effectively reducing it by 90%.




In [9]:
base_model = advanced_training_options.AbaloneNet(num_feats)
base_model.to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(params=base_model.parameters(), lr=0.01)
scheduler = advanced_training_options.create_scheduler(optimizer)

m_res, hist, b_ep = advanced_training_options.basic_training_loop(base_model,train_loader, loss_function, optimizer, 
                                                                  patience=30, min_delta=0.001, val_loader=val_loader,
                                                                  scheduler=scheduler)

Epoch 25/500, Train Loss: 4.3788, Validation Loss: 4.7765, Validation MAE: 1.5557
Epoch 50/500, Train Loss: 4.2886, Validation Loss: 4.7452, Validation MAE: 1.5096
Stopping at epoch 70. Early stopping triggered after 30 epochs.


In [10]:
b_ep

40

In [11]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res,x_test,y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)

Evaluation
MSE:  4.0506
RMSE: 2.0126 rings
MAE:  1.4308 rings
R^2:  0.6054


,actual_rings,predicted_rings
0,8.0,9.0
1,11.0,10.0
2,22.0,22.0
3,7.0,7.0
4,9.0,10.0
5,10.0,13.0
6,13.0,12.0
7,6.0,8.0
8,9.0,10.0
9,8.0,9.0


# Dropout Regularization

Regularization techniques are designed to ensure our model strikes a balance between bias (underfitting) and variance (overfitting). Regularization methods usually work by adding a penalty on the complexity of the model, effectively preventing the model from learning too much noise.

One such regularization technique is 'Dropout'. Instead of adding a penalty to the loss function, Dropout randomly disables a fraction of neurons at random (typically ranging from 20% to 50%), effectively creating different architectures of the same network for each training instance. This sorto of looks like an ensemble of networks.

In [12]:
base_model = advanced_training_options.AbaloneNetDropout(num_feats, dropout_probability=0.3)
base_model.to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(params=base_model.parameters(), lr=0.01)
scheduler = advanced_training_options.create_scheduler(optimizer)

m_res_dropout, hist, b_ep = advanced_training_options.basic_training_loop(base_model,train_loader, loss_function, optimizer, 
                                                                  patience=30, min_delta=0.001, val_loader=val_loader,
                                                                  scheduler=scheduler)

Epoch 25/500, Train Loss: 7.3708, Validation Loss: 4.9221, Validation MAE: 1.5584
Stopping at epoch 41. Early stopping triggered after 30 epochs.


In [13]:
b_ep

11

In [14]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res_dropout,x_test,y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)

Evaluation
MSE:  4.4443
RMSE: 2.1081 rings
MAE:  1.4747 rings
R^2:  0.5670


,actual_rings,predicted_rings
0,8.0,8.0
1,11.0,10.0
2,22.0,18.0
3,7.0,7.0
4,9.0,10.0
5,10.0,12.0
6,13.0,12.0
7,6.0,8.0
8,9.0,11.0
9,8.0,9.0


# L1 and L2 Regularization

L1 and L2 regularization are ways to make a neural network less likely to overfit by adding a penalty to large weights. 

Suppose the usual regression loss is the mean squared error:

loss = MSE(predictions, targets)

With regularization we train with an augmented loss:

regularized_loss = MSE + penalty

The two common penalties are:

- **L1 regularization** adds the sum of the absolute values of the weights. It can push some weights very close to zero, which can make the model simpler.
- **L2 regularization** adds the sum of the squared weights. It discourages very large weights and usually makes the model smoother.

In PyTorch, L2 regularization is usually implemented through the optimizer argument weight_decay. L1 regularization is not built into most optimizers, so we add it manually to the loss before calling loss.backward().


In [15]:
base_model = advanced_training_options.AbaloneNet(num_feats)
base_model.to(device)
loss_function = nn.MSELoss()
l2_lambda = 1e-4
l1_lambda = 1e-5
optimizer = optim.Adam(params=base_model.parameters(), lr=0.01, weight_decay=l2_lambda)
scheduler = advanced_training_options.create_scheduler(optimizer)

m_res, hist, b_ep = advanced_training_options.basic_training_loop(base_model,train_loader, loss_function, optimizer, 
                                                                  patience=30, min_delta=0.001, val_loader=val_loader,
                                                                  scheduler=scheduler, l1_lambda=l1_lambda)

Epoch 25/500, Train Loss: 4.4184, Validation Loss: 5.0832, Validation MAE: 1.6601
Stopping at epoch 48. Early stopping triggered after 30 epochs.


In [16]:
b_ep

18

In [17]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res_dropout,x_test,y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)

Evaluation
MSE:  4.4443
RMSE: 2.1081 rings
MAE:  1.4747 rings
R^2:  0.5670


,actual_rings,predicted_rings
0,8.0,8.0
1,11.0,10.0
2,22.0,18.0
3,7.0,7.0
4,9.0,10.0
5,10.0,12.0
6,13.0,12.0
7,6.0,8.0
8,9.0,11.0
9,8.0,9.0


# Batch Normalization

Batch Normalization is a technique used in neural networks to normalize the inputs to each layer by adjusting and scaling the activations. For each mini-batch, it computes the mean and variance of the inputs, then normalizes them by subtracting the mean and dividing by the standard deviation, followed by learnable scaling and shifting parameters.

This helps solve the problem of internal covariate shift, where the distribution of layer inputs changes during training, making optimization difficult. It stabilizes training, allows for higher learning rates, reduces the need for careful initialization, and acts as a form of regularization to prevent overfitting.

In PyTorch, the `nn.BatchNorm1d(num_features)` layer requires the `num_features` parameter, which specifies the number of features in the input (e.g., 64 for a fully connected layer with 64 neurons). It differs from `nn.BatchNorm2d(num_features)`, which is used for 2D inputs like those from convolutional layers, where `num_features` refers to the number of channels, and normalization is applied across the spatial dimensions (height and width) for each channel.

In [4]:
base_model = advanced_training_options.AbaloneNetBatchNorm(num_feats)
base_model.to(device)
loss_function = nn.MSELoss()
l2_lambda = 1e-4
l1_lambda = 1e-5
optimizer = optim.Adam(params=base_model.parameters(), lr=0.01, weight_decay=l2_lambda)
scheduler = advanced_training_options.create_scheduler(optimizer)

m_res, hist, b_ep = advanced_training_options.basic_training_loop(base_model,train_loader, loss_function, optimizer, 
                                                                  patience=30, min_delta=0.001, val_loader=val_loader,
                                                                  scheduler=scheduler, l1_lambda=l1_lambda)

Epoch 25/500, Train Loss: 4.3954, Validation Loss: 5.0887, Validation MAE: 1.6144
Epoch 50/500, Train Loss: 4.3457, Validation Loss: 4.8776, Validation MAE: 1.5305
Stopping at epoch 60. Early stopping triggered after 30 epochs.


In [5]:
b_ep

30

In [7]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res,x_test,y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)

Evaluation
MSE:  4.2127
RMSE: 2.0525 rings
MAE:  1.4693 rings
R^2:  0.5896


,actual_rings,predicted_rings
0,8.0,9.0
1,11.0,10.0
2,22.0,21.0
3,7.0,8.0
4,9.0,10.0
5,10.0,13.0
6,13.0,12.0
7,6.0,10.0
8,9.0,10.0
9,8.0,9.0


# Bayesian Hyperparameter Optimization for PyTorch

Bayesian Hyperparameter Optimization is a method of finding hyperparameters more efficiently than a grid search. Because each candidate set of hyperparameters requires a retraining of the neural network, it is best to keep the number of candidate sets to a minimum. Bayesian Hyperparameter Optimization achieves this by training a model to predict good candidate sets of hyperparameters. 

Bayesian optimization works on a vector of numbers, not on a problematic notion like how many layers and neurons are on each layer. To represent the neuron structure as a vector, we use several numbers.

- **dropout** - The dropout percent for each layer.
- **neuronPct** - What percent of our fixed 5,000 maximum number of neurons do we wish to use? This parameter specifies the total count of neurons in the entire network.
- **neuronShrink** - Neural networks usually start with more neurons on the first hidden layer and then decrease this count for additional layers. This percent specifies how much to shrink subsequent layers based on the previous layer. We stop adding more layers once we run out of neurons (the count specified by neuronPct).

These three numbers define the structure of the neural network. The commends in the below code show exactly how the program constructs the network.


In [18]:
# We optimize a flexible network with dropout.
# You can also try: advanced_training_options.AbaloneNet or advanced_training_options.AbaloneNetDropout
model_choice = "bayes_dropout"

bayes_objective, bayes_trials = advanced_training_options.bayesian_objective_factory(
    model_choice,
    num_feats,
    x_train,
    y_train,
    x_val,
    y_val,
    device,
    max_epochs=80,
)

bayes_optimizer = advanced_training_options.run_bayesian_optimization(
    bayes_objective,
    init_points=5,
    n_iter=10,
)


|   iter    |  target   |  dropout  | neuron... | neuron... |  log_lr   |  log_l1   |  log_l2   | batch_... |
-------------------------------------------------------------------------------------------------------------
Stopping at epoch 41. Early stopping triggered after 12 epochs.
| 1         | -2.147014 | 0.1498160 | 0.9605714 | 0.7391963 | -2.802683 | -7.219906 | -7.220027 | 37.576026 |
Stopping at epoch 74. Early stopping triggered after 12 epochs.
| 2         | -2.394732 | 0.3464704 | 0.6808920 | 0.7248435 | -3.958831 | -3.150450 | -3.837786 | 52.384554 |
Stopping at epoch 59. Early stopping triggered after 12 epochs.
| 3         | -2.161215 | 0.0727299 | 0.3467236 | 0.4825453 | -2.950487 | -5.840274 | -6.543854 | 90.737877 |
| 4         | -2.156911 | 0.0557975 | 0.4337157 | 0.5198171 | -3.087860 | -4.074120 | -7.001631 | 81.366506 |
| 5         | -2.745253 | 0.2369658 | 0.2371603 | 0.6645269 | -3.658951 | -7.674742 | -3.255572 | 124.70067 |
Stopping at epoch 61. Early stopping t

In [19]:
bayes_results = pd.DataFrame(bayes_trials).sort_values("val_rmse")
display(bayes_results)

best_params = bayes_results.iloc[0].to_dict()
best_params["batch_size"] = int(best_params["batch_size"])
best_params


,dropout,neuron_pct,neuron_shrink,lr,l1_lambda,l2_lambda,batch_size,hidden_sizes,best_epoch,val_rmse
5,0.076152,0.261065,0.495260,0.005984,4.205871e-07,3.435824e-08,32,"[35, 18, 9, 4]",49,2.127894
8,0.000000,1.000000,0.900000,0.010000,1.000000e-03,1.000000e-08,128,"[74, 67, 60, 54]",26,2.145520
0,0.149816,0.960571,0.739196,0.001575,6.026889e-08,6.025216e-08,32,"[91, 67, 50, 37]",29,2.147014
3,0.055798,0.433716,0.519817,0.000817,8.431014e-05,9.962513e-08,64,"[57, 30, 16, 8]",79,2.156912
10,0.000000,1.000000,0.900000,0.010000,1.000000e-08,8.432927e-05,64,"[74, 67, 60, 54]",15,2.159069
2,0.072730,0.346724,0.482545,0.001121,1.444525e-06,2.858549e-07,64,"[48, 23, 11, 5]",47,2.161216
14,0.000000,1.000000,0.900000,0.010000,1.000000e-08,4.996162e-07,64,"[74, 67, 60, 54]",22,2.162379
6,0.400000,1.000000,0.900000,0.000100,1.000000e-03,1.000000e-03,32,"[74, 67, 60, 54]",28,2.300540
1,0.346470,0.680892,0.724844,0.000110,7.072114e-04,1.452825e-04,64,"[66, 48, 35, 25]",62,2.394733
12,0.400000,1.000000,0.900000,0.000100,1.000000e-08,1.000000e-03,64,"[74, 67, 60, 54]",49,2.414349


{'dropout': 0.07615222176221137,
 'neuron_pct': 0.26106470432239115,
 'neuron_shrink': 0.4952599525548401,
 'lr': 0.0059842256811650455,
 'l1_lambda': 4.2058714453395575e-07,
 'l2_lambda': 3.4358236788777586e-08,
 'batch_size': 32,
 'hidden_sizes': [35, 18, 9, 4],
 'best_epoch': 49,
 'val_rmse': 2.1278937215509766}

In [20]:
x_dev = torch.cat([x_train, x_val], dim=0)
y_dev = torch.cat([y_train, y_val], dim=0)

b_ep = int(best_params["best_epoch"])

m_res_bayes, hist, b_ep = advanced_training_options.train_final_bayesian_model(
    model_choice,
    best_params,
    num_feats,
    x_dev,
    y_dev,
    device,
    epochs=b_ep,
)


In [21]:
b_ep


49

In [22]:
metrics, result_df = advanced_training_options.evaluate_regression(m_res_bayes, x_test, y_test)
result_df['predicted_rings'] = round(result_df['predicted_rings'])
display(result_df)


Evaluation
MSE:  4.6529
RMSE: 2.1571 rings
MAE:  1.4918 rings
R^2:  0.5467


,actual_rings,predicted_rings
0,8.0,9.0
1,11.0,9.0
2,22.0,20.0
3,7.0,7.0
4,9.0,10.0
5,10.0,13.0
6,13.0,11.0
7,6.0,9.0
8,9.0,11.0
9,8.0,9.0
